# Lab 2: Recommender Systems
## Rank Markets for Analyst Review

**Timebox:** 90 minutes across the pre-lunch and post-lunch blocks.

The recommender ranks fictional geographic market plays for analyst review. It does not score individuals or make eligibility, assignment, or outreach decisions.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/TheDeafOne/USARD.git"
REPO_DIR = Path("/content/USARD")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repository already cloned.")

%cd {REPO_DIR}


## Mission

Given the validated planning snapshot, identify which regions warrant additional human analysis for a fictional mission need. A useful recommendation must be constrained by data readiness, explainable, and evaluated as a ranked list.


In [ ]:
from pathlib import Path
import pandas as pd

root_candidates = (Path.cwd(), Path.cwd().parent)
ROOT = next((path for path in root_candidates if (path / 'data' / 'recommender').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run from the repository root or notebooks directory.')
DATA_DIR = ROOT / 'data' / 'recommender'
ARTIFACT_DIR = ROOT / 'artifacts' / 'recommender'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
markets = pd.read_csv(DATA_DIR / 'market_inputs.csv')
display(markets)


## Task 1: Validate recommendation inputs

A region is ready for the baseline model only when its data quality status is APPROVED and its source freshness is seven days or less. Return issues with region_id, rule, severity, and detail. Flag both an unapproved row and a stale row when both conditions apply.


In [ ]:
def validate_market_inputs(markets):
    # TODO: Return one issue for non-APPROVED data quality and one for freshness over seven days.
    raise NotImplementedError('Complete Task 1 before running this cell.')


In [ ]:
market_issues = validate_market_inputs(markets)
assert len(market_issues) == 2
assert set(market_issues['rule']) == {'market_not_ready', 'stale_source'}
assert set(market_issues['region_id']) == {'M-05'}
display(market_issues)
print('Task 1 checks passed.')


## Task 2: Score eligible markets

Create a transparent baseline score for eligible markets only. Normalize opportunity count and station capacity by their maximum among eligible markets, then use these weights:

- opportunity count: 0.30
- education fit: 0.25
- access score: 0.15
- historical appointment rate: 0.20
- station capacity: 0.10

Return all eligible rows with a market_score column.


In [ ]:
def score_markets(markets):
    # TODO: Filter to APPROVED rows with freshness of seven days or less.
    # TODO: Normalize opportunity_count and station_capacity within that eligible subset.
    # TODO: Compute and return market_score.
    raise NotImplementedError('Complete Task 2 before running this cell.')


In [ ]:
scored_markets = score_markets(markets)
assert len(scored_markets) == 5
assert scored_markets['market_score'].between(0, 1).all()
assert scored_markets.loc[scored_markets['region_id'].eq('M-01'), 'market_score'].iloc[0] > 0.8
print('Task 2 checks passed.')


## Task 3: Rank and explain

Sort markets by descending score and add a concise explanation naming the region's opportunity count, education fit, and current capacity. An explanation should reveal evidence, not conceal the calculation.


In [ ]:
def rank_and_explain(scored_markets):
    # TODO: Sort by market_score descending and add a recommendation_explanation column.
    raise NotImplementedError('Complete Task 3 before running this cell.')


In [ ]:
recommendations = rank_and_explain(scored_markets)
assert recommendations['region_id'].iloc[0] == 'M-01'
assert 'Northside' in recommendations['recommendation_explanation'].iloc[0]
display(recommendations[['region_id', 'region_name', 'market_score', 'recommendation_explanation']])
print('Task 3 checks passed.')


## Task 4: Evaluate the ranked list

Ordinary accuracy is not enough for a recommender. Calculate precision at k using domain_review_label as a fictional stand-in for a prior domain-expert review. Use the top three recommendations.


In [ ]:
def precision_at_k(recommendations, k):
    # TODO: Return the mean domain_review_label in the first k ranked rows.
    raise NotImplementedError('Complete Task 4 before running this cell.')


In [ ]:
precision = precision_at_k(recommendations, 3)
assert round(precision, 3) == 0.667
print(f'Precision at 3: {precision:.1%}')


## Handoff

Persist a review queue with quality constraints and explanations. The RAG lab will use approved documents to provide evidence for this plan; it will not replace these data checks.


In [ ]:
handoff_columns = ['region_id', 'region_name', 'market_score', 'recommendation_explanation', 'source_freshness_days', 'data_quality_status']
recommendations[handoff_columns].to_csv(ARTIFACT_DIR / 'market_recommendations.csv', index=False)
market_issues.to_csv(ARTIFACT_DIR / 'market_input_issues.csv', index=False)
display(recommendations[handoff_columns])
print(f'Wrote recommender artifacts to: {ARTIFACT_DIR}')


## Debrief

1. Which feature is a proxy rather than a direct measure of mission value?
2. What could make a high-scoring market inappropriate for action?
3. Which decision should remain with an analyst even if the ranking is stable?

### Optional extension

Change one weight and measure ranking stability. Explain whether the change reflects a real mission requirement or merely a modeling preference.


<!-- usard-next-colab-link -->
## Continue to Lab 3

[Open the next notebook in Colab](https://colab.research.google.com/github/TheDeafOne/USARD/blob/main/notebooks/03_rag_assistant.ipynb).
